In [ ]:
# ===== S-ARM SCALE — CONFIG (edit ONLY this cell) =========================
# Scout-then-scale: agent scouts at ≤1k picked L+w·S; THIS notebook raises
# budget on Colab multi-CPU. Hand the jsonl (+ .md) back into the repo.
#
# Winner context (campaign_12h + scouts):
#   - hard recoveries: s24 led; S-grid holdout peak near w=20
#   - default ARMS below: baseline + s20 + s24 (+ optional recommended)

REPO_URL   = "https://github.com/Avi161/ACSolverX.git"
REPO_DIR   = "ACSolverX"
# BRANCH must exist on the remote (this work lives on the campaign branch
# until merged into research/w5/stable-ac-escape).
BRANCH     = "cursor/heur-12h-anti-overfit-a42e"
CLONE      = True
UPDATE_REPO = True          # Restart → Run All picks up pushed .py fixes

MOUNT_DRIVE = True
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx/hsearch_s_scale"  # NEVER /content/drive alone

cfg = dict(
    # ---- what to run --------------------------------------------------
    # "bench66" | "unsolved124"
    DATASET   = "unsolved124",
    SUBSET    = None,        # None = all; int = first N

    # ---- arms (registered in run_s24_scale) ---------------------------
    # baseline = length-only control; s20 = S-grid holdout peak; s24 = hard leader
    ARMS      = ["baseline", "s20", "s24"],

    # ---- multi-session Colab (stride chunking) ------------------------
    # Spin N Colab VMs: set CHUNKS=N and CHUNK_INDEX=1..N on each.
    # Within one VM, N_WORKERS parallelizes across CPUs (result-neutral).
    CHUNKS       = 1,
    CHUNK_INDEX  = None,     # None = all rows in this session; 1..CHUNKS = one shard
    N_WORKERS    = "auto",

    # ---- budget (Colab production — agent stays ≤1000) ----------------
    # One run to NODE_BUDGET yields every CHECKPOINT below via solved_at.
    NODE_BUDGET = 100_000,
    CHECKPOINTS = [1_000, 5_000, 10_000, 25_000, 50_000, 100_000],

    # Cap: campaign used 48; scout_s_cap_addon.md says whether 24 is enough.
    MAX_RELATOR_LENGTH = 48,

    ENGINE    = "hcompact",
    KEEP_PATH = True,
    RESUME    = True,
    OUT_STEM  = "hsearch_s_scale",
)

HEARTBEAT_SECS = 60
PROGRESS_SECS  = 300


In [ ]:
# ==================== SETUP (clone / pull / mount / purge) ================
import os, sys, subprocess, importlib

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

from experiments.heuristic_search.core.hsolve import greedy_search_h
from experiments.heuristic_search.runners import run_s24_scale as _rs
_ = greedy_search_h("xyx", "yx", 20, max_relator_length=32,
                    config=_rs.run_ab.ARMS["s24"])
print("kernels warm — setup done")


In [ ]:
# ==================== RUN =================================================
from experiments.heuristic_search.runners.run_s24_scale import run_s24_scale
run_s24_scale(cfg, out_dir=(DRIVE_DIR if (IN_COLAB and MOUNT_DRIVE) else "results/hsearch"),
              heartbeat_secs=HEARTBEAT_SECS, progress_secs=PROGRESS_SECS)
